In [1]:
import pandas as pd
import datasets
import numpy
import os
import transformers
import torch

In [2]:
#test_labels = pd.read_csv(r"C:\Users\amine\Downloads\DASCI\Projet Pro Com\Dataset Touché-20241011T082748Z-001\Dataset Touché\valueeval24\test-english\labels.tsv", encoding="utf-8", sep="\t", header=0)
#test_sentences=pd.read_csv(r"C:\Users\amine\Downloads\DASCI\Projet Pro Com\Dataset Touché-20241011T082748Z-001\Dataset Touché\valueeval24\test-english\sentences.tsv", encoding="utf-8", sep="\t", header=0)
train_labels= pd.read_csv(r".\Dataset Touché\valueeval24\training-english\labels.tsv", encoding="utf-8", sep="\t", header=0)
train_sentences= pd.read_csv(r".\Dataset Touché\valueeval24\training-english\labels.tsv", encoding="utf-8", sep="\t", header=0)



In [3]:
train_labels.head()
train_sentences.head()

,Text-ID,Sentence-ID,Self-direction: thought attained,Self-direction: thought constrained,Self-direction: action attained,Self-direction: action constrained,Stimulation attained,Stimulation constrained,Hedonism attained,Hedonism constrained,...,Benevolence: caring attained,Benevolence: caring constrained,Benevolence: dependability attained,Benevolence: dependability constrained,Universalism: concern attained,Universalism: concern constrained,Universalism: nature attained,Universalism: nature constrained,Universalism: tolerance attained,Universalism: tolerance constrained
0,BG_002,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,BG_002,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,BG_002,3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,BG_002,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,BG_002,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
values = [ "Self-direction: thought", "Self-direction: action", "Stimulation",  "Hedonism", "Achievement", "Power: dominance", "Power: resources", "Face", "Security: personal", "Security: societal", "Tradition", "Conformity: rules", "Conformity: interpersonal", "Humility", "Benevolence: caring", "Benevolence: dependability", "Universalism: concern", "Universalism: nature", "Universalism: tolerance" ]
labels = sum([[value + " attained", value + " constrained"] for value in values], [])

In [5]:
from transformers import DebertaTokenizer, DebertaForSequenceClassification, Trainer, TrainingArguments

# Charger le tokenizer et le modèle DeBERTa
model_name = "microsoft/deberta-base"
tokenizer = DebertaTokenizer.from_pretrained(model_name)



In [6]:
# Create torch dataset
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

In [7]:
def load_dataset(directory, tokenizer, load_labels=True):
    sentences_file_path = os.path.join(directory, "sentences.tsv")
    labels_file_path = os.path.join(directory, "labels.tsv")
    
    data_frame = pd.read_csv(sentences_file_path, encoding="utf-8", sep="\t", header=0)
    X=data_frame["Text"].to_list()
    encoded_sentences = tokenizer(X,padding=True, truncation=True, max_length=512)

    if load_labels and os.path.isfile(labels_file_path):
        labels_frame = pd.read_csv(labels_file_path, encoding="utf-8", sep="\t", header=0)
        labels_frame = pd.merge(data_frame, labels_frame, on=["Text-ID", "Sentence-ID"])
        labels_matrix = numpy.zeros((labels_frame.shape[0], len(labels)))
        for idx, label in enumerate(labels):
            if label in labels_frame.columns:
                labels_matrix[:, idx] = (labels_frame[label] >= 0.5).astype(int)
        lbls = labels_matrix.tolist()
        
    encoded_sentences=Dataset(encoded_sentences,lbls)

    #encoded_sentences = datasets.Dataset.from_dict(encoded_sentences)
    
    return encoded_sentences, data_frame["Text-ID"].to_list(), data_frame["Sentence-ID"].to_list()

In [8]:
#directory_test=r"C:\Users\amine\Downloads\DASCI\Projet Pro Com\Dataset Touché-20241011T082748Z-001\Dataset Touché\valueeval24\test-english"
directory_train=r".\Dataset Touché\valueeval24\training-english"
directory_validation=r".\Dataset Touché\valueeval24\validation-english"

#encoded_sentences_test, text_ids_test, sentence_ids_test = load_dataset(directory_test, tokenizer)
encoded_sentences_train, text_ids_train, sentence_ids_train = load_dataset(directory_train, tokenizer)
encoded_sentences_validation, text_ids_validation, sentence_ids_validation = load_dataset(directory_validation, tokenizer)

In [9]:
encoded_sentences_validation[5]

{'input_ids': tensor([    1, 46688,   596,  3970,   132,     4,   245,   153,  1236, 10155,
            21,   278,    25,     5,  1002,    13,   769, 12211,     7,   555,
            41,  1973,     6,  2784,  1793,  7384,    26,     5,   247,  1395,
            28, 14015,   137,    70,   167,    81,     5,  1046,     9,  3620,
            33,    57, 26999,     4,     2,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,   

In [10]:
num_labels = len(labels)
model = DebertaForSequenceClassification.from_pretrained(model_name, num_labels=num_labels) 
for param in model.parameters():
    param.requires_grad=False  

Some weights of DebertaForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
import torch.nn as nn
model.classifier = nn.Sequential(
    nn.ReLU(),
    nn.Dropout(0.3),  # Dropout pour éviter le surapprentissage
    nn.Linear(model.config.hidden_size, len(labels)) 
)

In [12]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

In [ ]:
#X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2,stratify=y)


In [23]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(p):
    # Unpack predictions and labels
    preds, labels = p
    # Apply a threshold to predict label presence
    preds = (preds >= 0.5).astype(int)  # Adjust the threshold as needed

    # Calculate multi-label metrics
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='samples', zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
#from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import Trainer, TrainingArguments

#def compute_metrics(pred):
    labels = pred.label_ids
    # Prendre les probabilités prédictives et les convertir en labels binaires avec un seuil de 0.5
    preds = (pred.predictions >= 0.5).astype(int)
    
    # Calcul des métriques pour un problème multi-label
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='samples',zero_division=0)
    accuracy = accuracy_score(labels, preds)
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:

#from transformers import DataCollatorWithPadding

# Créer un collateur de données qui va appliquer le padding aux séquences
#data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
#from transformers import TrainingArguments

#training_args = TrainingArguments(
    output_dir="./results",                   
    num_train_epochs=3,                        
    per_device_train_batch_size=16,           
    per_device_eval_batch_size=16,            
    evaluation_strategy="epoch",               
)

c:\Users\amine\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [24]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="output",
    save_steps=100,
    num_train_epochs=3,
    per_device_train_batch_size=8

)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded_sentences_train,
    eval_dataset=encoded_sentences_validation,
    compute_metrics=compute_metrics
)

In [19]:
model

DebertaForSequenceClassification(
  (deberta): DebertaModel(
    (embeddings): DebertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=0)
      (LayerNorm): DebertaLayerNorm()
      (dropout): StableDropout()
    )
    (encoder): DebertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaLayer(
          (attention): DebertaAttention(
            (self): DisentangledSelfAttention(
              (in_proj): Linear(in_features=768, out_features=2304, bias=False)
              (pos_dropout): StableDropout()
              (pos_proj): Linear(in_features=768, out_features=768, bias=False)
              (pos_q_proj): Linear(in_features=768, out_features=768, bias=True)
              (dropout): StableDropout()
            )
            (output): DebertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): DebertaLayerNorm()
              (dropout): StableDropout()
            )
          )
          (

In [26]:
trainer.train(resume_from_checkpoint=True)


c:\Users\amine\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\trainer.py:3354: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(

  0%|          | 0/16785 [00:00<?, ?it/s]

{'train_runtime': 0.0102, 'train_samples_per_second': 13207306.77, 'train_steps_per_second': 1650987.117, 'train_loss': 0.0, 'epoch': 3.0}


TrainOutput(global_step=16785, training_loss=0.0, metrics={'train_runtime': 0.0102, 'train_samples_per_second': 13207306.77, 'train_steps_per_second': 1650987.117, 'total_flos': 2.991948439575312e+16, 'train_loss': 0.0, 'epoch': 3.0})

In [27]:
trainer.evaluate()

  0%|          | 0/1863 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
#trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_sentences_train,  
    eval_dataset=encoded_sentences_test,   
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


#trainer.train(resume_from_checkpoint=True)


c:\Users\amine\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\trainer.py:3262: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(

  0%|          | 0/8394 [00:00<?, ?it/s]

c:\Users\amine\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\trainer.py:2944: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint_rng_state = t

{'loss': 1.5166, 'grad_norm': 9.793636322021484, 'learning_rate': 2.9151775077436266e-05, 'epoch': 1.25}
{'loss': 1.4676, 'grad_norm': 12.785459518432617, 'learning_rate': 2.617345723135573e-05, 'epoch': 1.43}
{'loss': 1.4761, 'grad_norm': 12.633447647094727, 'learning_rate': 2.3195139385275197e-05, 'epoch': 1.61}
{'loss': 1.4546, 'grad_norm': 7.605840682983398, 'learning_rate': 2.0216821539194662e-05, 'epoch': 1.79}
{'loss': 1.4433, 'grad_norm': 5.5909905433654785, 'learning_rate': 1.723850369311413e-05, 'epoch': 1.97}


  0%|          | 0/911 [00:00<?, ?it/s]

{'eval_loss': 1.1324130296707153, 'eval_accuracy': 6.863889079552475e-05, 'eval_f1': 0.1081648111455506, 'eval_precision': 0.062126328038538896, 'eval_recall': 0.4819445397762372, 'eval_runtime': 1055.7777, 'eval_samples_per_second': 13.799, 'eval_steps_per_second': 0.863, 'epoch': 2.0}
{'loss': 1.2606, 'grad_norm': 8.94701862335205, 'learning_rate': 1.4260185847033596e-05, 'epoch': 2.14}
{'loss': 1.1401, 'grad_norm': 12.342790603637695, 'learning_rate': 1.1281868000953061e-05, 'epoch': 2.32}
{'loss': 1.1183, 'grad_norm': 8.773736953735352, 'learning_rate': 8.303550154872528e-06, 'epoch': 2.5}
{'loss': 1.1588, 'grad_norm': 13.732094764709473, 'learning_rate': 5.325232308791995e-06, 'epoch': 2.68}
{'loss': 1.1213, 'grad_norm': 15.755253791809082, 'learning_rate': 2.3469144627114603e-06, 'epoch': 2.86}


  0%|          | 0/911 [00:00<?, ?it/s]

{'eval_loss': 1.0237795114517212, 'eval_accuracy': 0.0, 'eval_f1': 0.11307263118885702, 'eval_precision': 0.06528388291721396, 'eval_recall': 0.48595419498020914, 'eval_runtime': 1061.5412, 'eval_samples_per_second': 13.724, 'eval_steps_per_second': 0.858, 'epoch': 3.0}
{'train_runtime': 26889.1107, 'train_samples_per_second': 4.994, 'train_steps_per_second': 0.312, 'train_loss': 0.8369546798913287, 'epoch': 3.0}


TrainOutput(global_step=8394, training_loss=0.8369546798913287, metrics={'train_runtime': 26889.1107, 'train_samples_per_second': 4.994, 'train_steps_per_second': 0.312, 'total_flos': 5434714756976520.0, 'train_loss': 0.8369546798913287, 'epoch': 3.0})

In [50]:
#test_metrics = trainer.evaluate(encoded_sentences_validation)
#print("Évaluation sur le jeu de test:", test_metrics)


In [49]:
#test_metrics = trainer.evaluate(encoded_sentences_validation)
#print("Évaluation sur le jeu de test:", test_metrics)
